# Abstention control on synthetic (U4s)

Registered in `docs/protocol.md` ("Registered before running: abstention control on synthetic (U4s)").
The single synthetic abstention run cut hallucination on evicted facts by 72.8 % with no control. This
trains the same LoRA on the compaction mask and on a random mask of the same size, five seeds each,
on the original corpus, and runs the registered paired t.


In [ ]:
import glob, os, subprocess, sys, zipfile

GIT_URL = 'https://github.com/rajul-kk/context-to-weights.git'
REPO = '/kaggle/working/myrios'
MARKER = 'baselines/cascading.py'

def looks_like_source(d):
    return os.path.exists(os.path.join(d, MARKER))

if not looks_like_source(REPO):
    src = None
    for d in sorted(glob.glob('/kaggle/input/*')):
        if looks_like_source(d):
            src = d
            break
        for z in sorted(glob.glob(os.path.join(d, '*.zip'))):
            os.makedirs(REPO, exist_ok=True)
            zipfile.ZipFile(z).extractall(REPO)
            if looks_like_source(REPO):
                src = REPO
                break
        if src:
            break
    if src and src != REPO:
        subprocess.run(['cp', '-r', src, REPO], check=True)
    if not looks_like_source(REPO) and GIT_URL:
        r = subprocess.run(['git', 'clone', GIT_URL, REPO], capture_output=True, text=True)
        print(r.stdout, r.stderr)
    assert looks_like_source(REPO), (
        'No source found. Either (a) run scripts/package_source.py locally, upload the zip '
        'as a Kaggle Dataset, and attach it via Add Input, or (b) set GIT_URL above. '
        f'Searched /kaggle/input/*, saw: {sorted(glob.glob("/kaggle/input/*"))}')

os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'peft', 'accelerate', 'datasets'])

try:
    import importlib.metadata as _md
    _v = _md.version('torchao')
    if tuple(int(x) for x in _v.split('.')[:2]) < (0, 16):
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'uninstall', '-y', 'torchao'])
        print(f'removed incompatible torchao {_v}')
except Exception:
    pass
exec(open('notebooks/_runner.py').read())
print('cwd', os.getcwd())
print('files', sorted(os.listdir('.'))[:10])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    print()
    print('=' * 68)
    print('NO GPU. Kaggle installed the CPU build of torch, so this session')
    print('has no accelerator attached. Everything below will be far too slow.')
    print()
    print('Fix: right panel -> Session options -> Accelerator -> GPU T4 x2,')
    print('then Run All again. The image swaps to a CUDA torch build on restart.')
    print('=' * 68)
else:
    print('gpu', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')



## Corpus and compaction


In [ ]:
import os, sys, json, shutil, glob
REPO = '/kaggle/working/myrios'
os.chdir(REPO); sys.path.insert(0, REPO)
if 'run' not in dir():
    exec(open('notebooks/_runner.py').read())

CFG = 'configs/kaggle_synthetic.yaml'
COMPACTOR = 'Qwen/Qwen2.5-1.5B-Instruct'
TARGET = 'Qwen/Qwen2.5-0.5B-Instruct'
SEEDS = [0, 1, 2, 3, 4]
OUT = '/kaggle/working/u4s'
os.makedirs(OUT, exist_ok=True)

from common.io import load_config
RUNS = load_config(CFG)['run_root']
CASC = f'{RUNS}/cascading'
run('python data/generate_synthetic.py --n-train 48 --n-eval 48 --n-turns 120')
run(f'python baselines/cascading.py --config {CFG} --split both --out {CASC} --set model.base={COMPACTOR}')
run(f'python eval/span_report.py --events {CASC}/eval_events.jsonl --show 0 --out {OUT}/span_report.json')


## Runs


In [ ]:
def eval_arm(label, adapter, dst):
    a = f'--adapter {adapter}' if adapter else ''
    run(f'python eval/abstention.py --config {CFG} --contexts {CASC}/eval_contexts.jsonl --label {label} '
        f'--out {dst}/{label} {a} --set model.base={TARGET}')

eval_arm('no-adapter', None, OUT)
for seed in SEEDS:
    dst = f'{OUT}/seed_{seed}'
    os.makedirs(dst, exist_ok=True)
    for mask in ('compaction', 'random'):
        rd = f'{RUNS}/abstain_{mask}_s{seed}'
        run(f'python sleep/abstain.py --config {CFG} --contexts {CASC}/train_contexts.jsonl --run-dir {rd} '
            f'--mask {mask} --batch-size 1 --grad-accum 8 --set model.base={TARGET} seed={seed}')
        eval_arm(mask, f'{rd}/adapter', dst)
    shutil.make_archive('/kaggle/working/u4s_runs', 'zip', OUT)
    rows = {m: json.load(open(f'{dst}/{m}.summary.json')) for m in ('compaction', 'random')}
    print(f'seed {seed}: ' + '  '.join(
        f"{m} halluc {r['hallucination_rate_evicted']:.3f} acc(ret) {r['accuracy_retained']:.3f} "
        f"abstain(ret) {r['abstain_rate_retained']:.3f}" for m, r in rows.items()), flush=True)


## Registered test


In [ ]:
from scipy import stats

base = json.load(open(f'{OUT}/no-adapter.summary.json'))
rows = [{m: json.load(open(f'{d}/{m}.summary.json')) for m in ('compaction', 'random')}
        for d in sorted(glob.glob(f'{OUT}/seed_*'))]
res = {'n_runs': len(rows), 'no_adapter': base}
for key in ('hallucination_rate_evicted', 'accuracy_retained', 'abstain_rate_retained', 'abstention_margin'):
    c = [r['compaction'][key] for r in rows]
    q = [r['random'][key] for r in rows]
    t, p = stats.ttest_rel(c, q)
    res[key] = {'compaction': sum(c) / len(c), 'random': sum(q) / len(q), 't': float(t), 'p': float(p)}
h = res['hallucination_rate_evicted']
acc_drop = base['accuracy_retained'] - res['accuracy_retained']['compaction']
below = h['p'] < 0.05 and h['t'] < 0
res['accuracy_retained_drop_vs_no_adapter'] = acc_drop
res['verdict'] = ('holds' if below and acc_drop <= 0.05 else
                  'buys abstention with refusals' if below else 'does not hold')
json.dump(res, open(f'{OUT}/test.json', 'w'), indent=1)
shutil.make_archive('/kaggle/working/u4s_runs', 'zip', OUT)
print(json.dumps({k: v for k, v in res.items() if k != 'no_adapter'}, indent=1))
